In [6]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [7]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

Environment: orbit_wars v1.0.9
Players: [2, 4]
Max steps: 500


In [8]:
# Run a quick game to see what the observation looks like
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

Player: 0
Angular velocity: 0.0447 rad/turn

Planets (28):
  id=0 owner=Neutral    pos=(84.5, 91.4) r=1.7 ships=50 prod=2
  id=1 owner=Neutral    pos=(8.6, 84.5) r=1.7 ships=50 prod=2
  id=2 owner=Neutral    pos=(91.4, 15.5) r=1.7 ships=50 prod=2
  id=3 owner=Neutral    pos=(15.5, 8.6) r=1.7 ships=50 prod=2
  id=4 owner=Player 0   pos=(67.8, 96.3) r=2.1 ships=13 prod=3
  id=5 owner=Neutral    pos=(3.7, 67.8) r=2.1 ships=22 prod=3


In [27]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        min_dist = float('inf')
        for t in targets:
            if(t.ships >= mine.ships):
                continue
            def touches_sun(x, y, a):
                cx, cy = 50, 50
                r = 10
                dx = math.cos(a)
                dy = math.sin(a)
                vx = cx - x
                vy = cy - y
            
                t = vx * dx + vy * dy
            
                if t < 0:
                    return False
            
                closest_x = x + t * dx
                closest_y = y + t * dy
            
                dist2 = (closest_x - cx)**2 + (closest_y - cy)**2
            
                return dist2 <= r * r
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            angle = math.atan2(t.y - mine.y, t.x - mine.x)
            if(touches_sun(mine.x, mine.y, angle)==True):
                continue
            moves.append([mine.id, angle, t.ships+1])    

    return moves

In [ ]:
env4 = make("orbit_wars", debug=True)
env4.run([nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper])

final = env4.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env4.render(mode="ipython", width=800, height=600)

In [ ]:
# Test it against the random agent
env = make("orbit_wars", debug=True)
env.run([nearest_planet_sniper, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="ipython", width=800, height=600)

In [ ]:
%%writefile main.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves